In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
DATA_PATH = '../src/clean_data.csv'
OUTPUT_PATH = '../src/'

RANDOM_STATE = 42

TRAIN_SIZE = 0.6
VAL_SIZE = 0.2
TEST_SIZE = 0.2

CLASSES = ['N', 'S', 'V', 'F']

In [3]:
df = pd.read_csv(DATA_PATH)

df['record_id'] = df['record_id'].astype(int)

signal_columns = [f'x_{i}' for i in range(256)]

assert all(col in df.columns for col in signal_columns)
assert df['class'].isin(CLASSES).all()

print(df.shape)
display(df.head())

(100676, 261)


,record_id,annotation_sample,type,x_0,x_1,x_2,x_3,x_4,x_5,x_6,...,x_248,x_249,x_250,x_251,x_252,x_253,x_254,x_255,class,anomaly
0,100,370,N,-0.012063,-0.011870,-0.012323,-0.013667,-0.015435,-0.016664,-0.016444,...,0.031740,0.030491,0.028161,0.025477,0.023071,0.021232,0.019926,0.018956,N,0
1,100,662,N,-0.007641,-0.004494,-0.002241,-0.001259,-0.001613,-0.002982,-0.004748,...,-0.002470,-0.002016,-0.001907,-0.003053,-0.005780,-0.009739,-0.014134,-0.018067,N,0
2,100,946,N,-0.017088,-0.015658,-0.014706,-0.014649,-0.015687,-0.017712,-0.020420,...,0.016567,0.011699,0.007760,0.005607,0.005315,0.006294,0.007737,0.009020,N,0
3,100,1231,N,-0.015886,-0.012787,-0.010961,-0.011102,-0.013383,-0.017297,-0.021721,...,0.008601,0.006186,0.004477,0.003652,0.003842,0.004924,0.006367,0.007337,N,0
4,100,1515,N,0.006733,0.007868,0.007679,0.006301,0.004140,0.001740,-0.000411,...,0.021668,0.021216,0.020155,0.018518,0.016384,0.013742,0.010553,0.006958,N,0


In [4]:
record_ids = sorted(df['record_id'].unique())

patient_groups = {
    record_id: 201 if record_id == 202 else record_id
    for record_id in record_ids
} # bo rekordy 201 i 202 pochodzą od tego samego pacjenta

unique_patients = sorted(set(patient_groups.values()))

print('Liczba rekordów:', len(record_ids))
print('Liczba grup pacjentów:', len(unique_patients))

Liczba rekordów: 44
Liczba grup pacjentów: 43


In [6]:
df['patient_group'] = df['record_id'].map(patient_groups)

patient_class_counts = pd.crosstab(
    df['patient_group'],
    df['class']
).reindex(
    columns=CLASSES,
    fill_value=0
)

display(patient_class_counts)

C:\Users\weron\AppData\Local\Temp\ipykernel_24940\3555634115.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['patient_group'] = df['record_id'].map(patient_groups)


class,N,S,V,F
patient_group,,,,
100,2237,33,1,0
101,1859,3,0,0
103,2081,2,0,0
105,2526,0,41,0
106,1507,0,520,0
108,1739,4,17,2
109,2491,0,38,2
111,2123,0,1,0
112,2536,2,0,0


In [22]:
MIN_CLASS_COUNTS = {
    'N': 1000,
    'S': 100,
    'V': 100,
    'F': 50
}

def has_enough_classes(patient_ids):
    counts = patient_class_counts.loc[patient_ids].sum()

    return all(
        counts[class_name] >= minimum
        for class_name, minimum in MIN_CLASS_COUNTS.items()
    )
    
def acceptable_anomaly_ratio(patient_ids):
    counts = patient_class_counts.loc[patient_ids].sum()

    anomaly_count = counts[['S', 'V', 'F']].sum()
    total_count = counts.sum()

    ratio = anomaly_count / total_count

    return 0.05 <= ratio <= 0.20

In [23]:
for seed in range(RANDOM_STATE, RANDOM_STATE + 1000):

    train_patients, temp_patients = train_test_split(
        unique_patients,
        test_size=0.4,
        random_state=seed
    )

    val_patients, test_patients = train_test_split(
        temp_patients,
        test_size=0.5,
        random_state=seed
    )

    if (
        has_enough_classes(train_patients)
        and has_enough_classes(val_patients)
        and has_enough_classes(test_patients)
        and acceptable_anomaly_ratio(train_patients)
        and acceptable_anomaly_ratio(val_patients)
        and acceptable_anomaly_ratio(test_patients)
    ):
        print(f'Znaleziono poprawny podział. Seed: {seed}')
        break

else:
    raise RuntimeError(
        'Nie znaleziono podziału spełniającego warunki.'
    )

Znaleziono poprawny podział. Seed: 402


In [24]:
train_ids = [
    rid for rid in record_ids
    if patient_groups[rid] in train_patients
]

val_ids = [
    rid for rid in record_ids
    if patient_groups[rid] in val_patients
]

test_ids = [
    rid for rid in record_ids
    if patient_groups[rid] in test_patients
]

In [25]:
train_df = df[df['record_id'].isin(train_ids)].copy()
val_df = df[df['record_id'].isin(val_ids)].copy()
test_df = df[df['record_id'].isin(test_ids)].copy()

In [26]:
assert set(train_patients).isdisjoint(val_patients)
assert set(train_patients).isdisjoint(test_patients)
assert set(val_patients).isdisjoint(test_patients)

In [27]:
assert (
    len(train_df) + len(val_df) + len(test_df)
    == len(df)
)

In [28]:
for dataset in [train_df, val_df, test_df]:
    assert set(dataset['class'].unique()) == set(CLASSES)

In [29]:
for name, dataset in [
    ('Train', train_df),
    ('Validation', val_df),
    ('Test', test_df)
]:
    print(f'\n{name}')
    print('Rekordy:', dataset['record_id'].nunique())
    print('Uderzenia:', len(dataset))
    print('Anomalie [%]:', round(dataset['anomaly'].mean() * 100, 2))


Train
Rekordy: 25
Uderzenia: 57203
Anomalie [%]: 8.57

Validation
Rekordy: 9
Uderzenia: 19250
Anomalie [%]: 13.96

Test
Rekordy: 10
Uderzenia: 24223
Anomalie [%]: 12.38


In [30]:
class_distribution = pd.DataFrame({
    'Train': train_df['class'].value_counts(),
    'Validation': val_df['class'].value_counts(),
    'Test': test_df['class'].value_counts()
}).reindex(CLASSES)

display(class_distribution)

,Train,Validation,Test
class,,,
N,52299,16563,21223
S,941,1526,314
V,3906,794,2308
F,57,367,378


In [31]:
class_distribution_pct = (
    class_distribution
    / class_distribution.sum(axis=0)
    * 100
).round(2)

display(class_distribution_pct)

,Train,Validation,Test
class,,,
N,91.43,86.04,87.62
S,1.65,7.93,1.30
V,6.83,4.12,9.53
F,0.10,1.91,1.56


In [32]:
train_df = train_df.drop(columns=['patient_group'])
val_df = val_df.drop(columns=['patient_group'])
test_df = test_df.drop(columns=['patient_group'])

In [33]:
train_df.to_csv(
    OUTPUT_PATH + 'train.csv',
    index=False
)

val_df.to_csv(
    OUTPUT_PATH + 'validation.csv',
    index=False
)

test_df.to_csv(
    OUTPUT_PATH + 'test.csv',
    index=False
)

In [34]:
split_records = pd.DataFrame(
    [(rid, 'train') for rid in train_ids]
    + [(rid, 'validation') for rid in val_ids]
    + [(rid, 'test') for rid in test_ids],
    columns=['record_id', 'split']
)

split_records = split_records.sort_values('record_id')

split_records.to_csv(
    OUTPUT_PATH + 'split_records.csv',
    index=False
)